# ChurnSense — Preprocessing & Feature Engineering

## Objective

The objective of this notebook is to prepare the Telco Customer Churn dataset for machine learning.

We will:
- Remove unnecessary identifier columns
- Convert the target variable into numerical form
- Separate numerical and categorical features
- Build a preprocessing pipeline
- Split the data using stratification
- Create engineered features

In [74]:
# Import all required libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn modules
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
import pickle

print("All libraries imported successfully!")

All libraries imported successfully!


In [75]:
df = pd.read_csv(r'C:\Users\USER\OneDrive\E&ICT,IIT,Guwahati,AI,ML,DS\Internship\Telco Customer Churn\Data\WA_Fn-UseC_-Telco-Customer-Churn.csv'
)
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [76]:
df = df.drop("customerID", axis=1)

In [77]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

In [78]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [79]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include="object").columns.tolist()

In [80]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [81]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [82]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [83]:
X_train["tenure_group"] = pd.cut(
    X_train["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"]
)

X_test["tenure_group"] = pd.cut(
    X_test["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"]
)

In [84]:
categorical_features.append("tenure_group")

In [85]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [86]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [87]:
X_train_processed.shape

(5634, 49)

In [88]:
import os
import sys

print("Current folder:", os.getcwd())

Current folder: c:\Users\USER\OneDrive\E&ICT,IIT,Guwahati,AI,ML,DS\Internship\2.Telco Customer Churn\notebook


In [89]:
import sys
import os

project_path = os.path.abspath("..")
sys.path.insert(0, project_path)

print("Project path:", project_path)
print("Files:", os.listdir(project_path))

Project path: c:\Users\USER\OneDrive\E&ICT,IIT,Guwahati,AI,ML,DS\Internship\2.Telco Customer Churn
Files: ['.gitignore', 'data', 'images', 'models', 'myenv', 'notebook', 'README.md', 'reports', 'requirements.txt']


In [90]:
import os

print(os.path.exists("../src"))
print(os.path.exists("../src/preprocessing.py"))

False
False


In [91]:
import os
print(os.getcwd())
print(os.listdir())

c:\Users\USER\OneDrive\E&ICT,IIT,Guwahati,AI,ML,DS\Internship\2.Telco Customer Churn\notebook
['01_data_exploration.ipynb', '02_eda.ipynb', '03_preprocessing.ipynb', 'best_model.pkl', 'scaler.pkl']


In [92]:
# Checking the skewness of numerical features
# Note: column name is 'Churn' (capital C) — Python is case-sensitive
skewness = df[df.select_dtypes(include=np.number).columns].skew().drop('Churn')  # Exclude the target variable
print("Skewness of numerical features:")
print(skewness)

Skewness of numerical features:
SeniorCitizen     1.833633
tenure            0.239540
MonthlyCharges   -0.220524
TotalCharges      0.963235
dtype: float64


In [93]:
# Separate features and target
X = df.drop(columns=['Churn'])
y = df['Churn']

# Train-test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size : {X_train.shape}")
print(f"Test set size     : {X_test.shape}")

Training set size : (5634, 19)
Test set size     : (1409, 19)


In [94]:
# Encode categorical columns before scaling
X_train_encoded = pd.get_dummies(X_train)
X_test_encoded  = pd.get_dummies(X_test)

# Align columns so both sets have the same features
X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded, join='left', axis=1, fill_value=0
)

# Feature scaling (only numeric data now)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled  = scaler.transform(X_test_encoded)
print("Feature scaling done!")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape : {X_test_scaled.shape}")

Feature scaling done!
X_train_scaled shape: (5634, 45)
X_test_scaled shape : (1409, 45)
